##### Copyright 2026 Google LLC.

In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Hybrid RAG with File Search and Google Search

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/hybrid_file_search_and_google_search.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

Most real assistants need two kinds of knowledge at once: **private knowledge** that lives
in your own documents, and **public knowledge** that lives on the open web and changes daily.
Gemini ships a grounding tool for each one:

*   [File Search](https://ai.google.dev/gemini-api/docs/file-search) is a managed RAG tool. You
    upload documents to a store and Gemini retrieves the relevant chunks at generation time.
*   [Google Search grounding](https://ai.google.dev/gemini-api/docs/google-search) connects the
    model to fresh, publicly available information from the web.

This notebook shows you two ways to combine them into a single *hybrid* pipeline:

1.  **Deterministic fallback routing** — you query File Search first, inspect what came back,
    and only fall back to Google Search when your documents cannot answer the question. You keep
    full control of the routing decision, which makes it auditable and cheap to reason about.
2.  **Native multi-tool routing** — you pass both tools in the same request and let Gemini pick
    the right one, or combine both in a single answer.

Along the way you will extract `grounding_metadata` from every response so you can show the user
exactly which document chunk or which web page an answer came from.

> **Note:** This notebook uses the `generateContent` API rather than the newer
> [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), because
> `grounding_metadata` (the citations that hybrid routing depends on) is returned on
> `generateContent` responses. The tool definitions themselves are the same in both APIs.

## Install dependencies

Install the [`google-genai`](https://pypi.org/project/google-genai) Python SDK.

In [2]:
%pip install -U -q "google-genai>=2.9.0"

## Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`.
If you don't already have an API key, or you're not sure how to create a Colab Secret, see
[Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb)
for a walkthrough.

**Important:** File Search associates uploaded documents with the API key's cloud project, so your
API key also grants access to everything you have uploaded. Keep it secret, and follow Google's
[API key best practices](https://support.google.com/googleapi/answer/6310037).

In [3]:
from google import genai
from google.colab import userdata
from google.genai import types

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

### Select a model

Grounding tools are supported across the current Gemini model family. Pick the one you want to use
throughout this guide.

In [4]:
MODEL_ID = "gemini-3.8-flash" # @param ["gemini-3.1-pro-preview", "gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite", "gemini-2.5-pro"] {"allow-input": true, "isTemplate": true}

## Create a sample document

To keep the notebook self-contained, you generate a small internal handbook for a fictional
company instead of downloading anything. Nothing in this document exists on the public web, so it
is a good stand-in for your own private corpus: the only way Gemini can answer questions about it
is by retrieving it from File Search.

In [5]:
HANDBOOK = """
Zephyr Robotics — Employee Handbook (Internal)

About the company
Company Name: Zephyr Robotics
Founded: 2019
Headquarters: San Francisco, CA

Time off
PTO Policy: Every full-time employee receives 20 days of paid time off per year.
Unused PTO does not roll over into the next calendar year.

Benefits
Health Insurance: Zephyr Robotics covers 90% of medical, dental, and vision premiums
for employees and their dependents.

Working arrangements
Remote Work: Zephyr Robotics runs a hybrid schedule of 3 days remote and 2 days in the
San Francisco office. Teams pick their own in-office days.
"""

with open("sample_handbook.txt", "w") as f:
    f.write(HANDBOOK)

print(HANDBOOK.strip())

Zephyr Robotics — Employee Handbook (Internal)

About the company
Company Name: Zephyr Robotics
Founded: 2019
Headquarters: San Francisco, CA

Time off
PTO Policy: Every full-time employee receives 20 days of paid time off per year.
Unused PTO does not roll over into the next calendar year.

Benefits
Health Insurance: Zephyr Robotics covers 90% of medical, dental, and vision premiums
for employees and their dependents.

Working arrangements
Remote Work: Zephyr Robotics runs a hybrid schedule of 3 days remote and 2 days in the
San Francisco office. Teams pick their own in-office days.


## Create a File Search store and upload the document

A File Search store holds your documents and their embeddings. Uploading kicks off a long-running
ingestion operation (chunking and embedding), so poll the operation until it reports `done`.

In [6]:
file_search_store = client.file_search_stores.create(
    config=types.CreateFileSearchStoreConfig(
        display_name="Zephyr Robotics Handbook Store"
    )
)

# Keep the store name around; every File Search call and the cleanup step needs it.
FILE_SEARCH_STORE_NAME = file_search_store.name
print(f"Created store: {FILE_SEARCH_STORE_NAME}")

Created store: fileSearchStores/zephyr-robotics-handbook-st-n0fqy4htnldi


In [7]:
import time

upload_op = client.file_search_stores.upload_to_file_search_store(
    file_search_store_name=FILE_SEARCH_STORE_NAME,
    file="sample_handbook.txt",
    config=types.UploadToFileSearchStoreConfig(
        display_name="Zephyr Robotics Employee Handbook",
    ),
)

print(f"Ingestion started: {upload_op.name}")

# Wait for chunking and embedding to finish before querying the store.
while not (upload_op := client.operations.get(upload_op)).done:
    time.sleep(2)
    print(".", end="")

print()
print("Ingestion complete.")

Ingestion started: fileSearchStores/zephyr-robotics-handbook-st-n0fqy4htnldi/upload/operations/zephyr-robotics-employee-ha-a383urf2chjj
.
Ingestion complete.


## Read citations from a response

Both grounding tools report their sources the same way, on
`response.candidates[0].grounding_metadata`. The difference is which field of each entry in
`grounding_chunks` is populated:

*   File Search fills in `chunk.retrieved_context` — the retrieved `text`, the document `title`,
    and often a `page_number`.
*   Google Search fills in `chunk.web` — the page `uri` and `title`.

Write one small helper that normalizes both shapes into plain dictionaries, and a second helper
that prints them. You will reuse these for both routing patterns.

In [8]:
def extract_citations(response):
    """Normalizes File Search and Google Search grounding chunks into dicts."""
    citations = []
    if response is None:
        return citations

    candidates = response.candidates or []
    if not candidates:
        return citations

    metadata = candidates[0].grounding_metadata
    if metadata is None:
        return citations

    for chunk in metadata.grounding_chunks or []:
        if chunk.retrieved_context is not None:
            snippet = chunk.retrieved_context.text or ""
            page_num = chunk.retrieved_context.page_number
            location = (
                f"page {page_num}" if page_num is not None else "retrieved chunk"
            )
            citations.append({
                "source_type": "document",
                "title": chunk.retrieved_context.title,
                "location": location,
                "snippet": snippet[:200],
            })
        elif chunk.web is not None:
            citations.append({
                "source_type": "web",
                "title": chunk.web.title,
                "location": chunk.web.uri,
                "snippet": "",
            })
    return citations


def show(label, response):
    """Prints a response together with its normalized citations."""
    if response is None:
        print("\n[no response returned]")
        return

    print(f"=== {label} ===")
    print((response.text or "").strip())

    citations = extract_citations(response)
    if not citations:
        print("\n[no grounding citations returned]")
        return

    print("\nSources:")
    for i, citation in enumerate(citations, start=1):
        print(f"  [{i}] ({citation['source_type']}) {citation['title']}")
        print(f"      {citation['location']}")
        if citation["snippet"]:
            print(f"      \"{citation['snippet']}...\"")

## Pattern 1: Deterministic fallback routing

In this pattern *your code* decides which source answers the question, in two steps:

1.  Ask Gemini with only the File Search tool enabled.
2.  Judge the result with a simple heuristic. If your documents clearly did not cover the
    question, re-ask with only the Google Search tool enabled.

The heuristic below stays deliberately basic — no scoring model, just two checks that are cheap
and easy to explain:

*   **No retrieval.** If `grounding_chunks` is empty, File Search found nothing relevant.
*   **Explicit non-answer.** If the model retrieved something but its reply says the documents do
    not contain the answer, treat that as insufficient too.

Deterministic routing is worth the extra call when you need to guarantee that private documents
are always consulted first, when you want to log the routing decision for audit, or when web
access has to be an explicit, observable escalation rather than a model choice.

In [9]:
NO_ANSWER_PHRASES = (
    "does not contain",
    "doesn't contain",
    "no information",
    "not mentioned",
    "not specified",
    "unable to answer",
    "cannot answer",
    "i don't know",
)


def is_sufficient(response):
    """Simple heuristic: did File Search actually ground this answer?"""
    if response is None:
        return False, "No response returned."

    citations = extract_citations(response)
    if not citations:
        return False, "File Search returned no chunks."

    text = (response.text or "").lower()
    if any(phrase in text for phrase in NO_ANSWER_PHRASES):
        return False, "The model reported that the documents do not cover the question."

    return True, f"Grounded in {len(citations)} document chunk(s)."


def ask_with_fallback(question, store_name=None, verbose=True):
    """Queries File Search first, then falls back to Google Search if needed."""
    store_name = store_name or FILE_SEARCH_STORE_NAME

    try:
        doc_response = client.models.generate_content(
            model=MODEL_ID,
            contents=question,
            config=types.GenerateContentConfig(
                tools=[types.Tool(
                    file_search=types.FileSearch(
                        file_search_store_names=[store_name]
                    )
                )],
            ),
        )
    except Exception as e:  # Surface API errors without stopping the notebook.
        print(f"File Search call failed: {type(e).__name__}: {e}")
        return None, "error"

    sufficient, reason = is_sufficient(doc_response)
    if verbose:
        print(f"Router: File Search -> {'sufficient' if sufficient else 'insufficient'}")
        print(f"        {reason}")

    if sufficient:
        return doc_response, "file_search"

    if verbose:
        print("Router: falling back to Google Search.")

    try:
        web_response = client.models.generate_content(
            model=MODEL_ID,
            contents=question,
            config=types.GenerateContentConfig(
                tools=[types.Tool(google_search=types.GoogleSearch())],
            ),
        )
    except Exception as e:
        print(f"Google Search call failed: {type(e).__name__}: {e}")
        return None, "error"

    return web_response, "google_search"

### Question 1: answered by your documents

The PTO policy only exists in the handbook, so File Search should return chunks and the router
should stop after step 1.

In [10]:
question_1 = "What is Zephyr Robotics' PTO policy?"

response_1, route_1 = ask_with_fallback(question_1)
print()

if response_1 is not None:
    show(f"Route: {route_1}", response_1)

Router: File Search -> sufficient
        Grounded in 1 document chunk(s).

=== Route: file_search ===
According to the Zephyr Robotics Employee Handbook, the Paid Time Off (PTO) policy is as follows:

* **Annual PTO Allowance:** Full-time employees receive **20 days** of paid time off per year.
* **Rollover Policy:** Unused PTO **does not roll over** into the following calendar year (use-it-or-lose-it).

Sources:
  [1] (document) Zephyr Robotics Employee Handbook
      page None
      "
Zephyr Robotics — Employee Handbook (Internal)

About the company
Company Name: Zephyr Robotics
Founded: 2019
Headquarters: San Francisco, CA

Time off
PTO Policy: Every full-time employee receives 2..."


### Question 2: not answered by your documents

Nothing in the handbook mentions macroeconomics, so the heuristic should mark the File Search
attempt as insufficient and escalate to Google Search. Notice that the citations that come back
are web pages rather than document chunks.

In [11]:
question_2 = "What is the current US inflation rate?"

response_2, route_2 = ask_with_fallback(question_2)
print()

if response_2 is not None:
    show(f"Route: {route_2}", response_2)

File Search call failed: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}



## Pattern 2: Native multi-tool routing

Instead of routing in your own code, hand Gemini both tools in the same request and let it decide.
The model reads the question, calls whichever tool fits, and can call both when a question spans
private and public knowledge.

This costs one request instead of up to two, and it handles blended questions that deterministic
routing would have to split by hand. The trade-off is that the routing decision now lives inside
the model, so you learn what happened by reading `grounding_metadata` after the fact rather than
by deciding it up front.

In [12]:
HYBRID_TOOLS = [
    types.Tool(
        file_search=types.FileSearch(
            file_search_store_names=[FILE_SEARCH_STORE_NAME]
        )
    ),
    types.Tool(google_search=types.GoogleSearch()),
]


def ask_hybrid(question):
    """Passes both grounding tools and lets Gemini choose."""
    try:
        return client.models.generate_content(
            model=MODEL_ID,
            contents=question,
            config=types.GenerateContentConfig(tools=HYBRID_TOOLS),
        )
    except Exception as e:
        print(f"Hybrid call failed: {type(e).__name__}: {e}")
        return None

Ask the same two questions again. The answers should match what Pattern 1 produced, but each one
now takes a single request.

In [13]:
for question in (question_1, question_2):
    response = ask_hybrid(question)
    if response is not None:
        show(question, response)
    print()

Hybrid call failed: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

Hybrid call failed: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'

### A question that needs both sources

Because both tools are live at once, the model can ground one part of an answer in your handbook
and another part on the web. Ask something that deliberately straddles the two.

In [14]:
blended_question = (
    "Zephyr Robotics covers 90% of health insurance premiums. "
    "How does that compare to the average employer contribution in the US?"
)

blended_response = ask_hybrid(blended_question)
if blended_response is not None:
    show("Blended question", blended_response)

Hybrid call failed: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}


## Inspect the raw grounding metadata

The `show` helper flattens citations for readability, but it is worth looking at the raw object
once so you know what else is available. Beyond `grounding_chunks`, the metadata carries:

*   `grounding_supports` — spans of the generated text mapped back to the chunk indices that
    support them, which is what you need to render inline footnote markers.
*   `web_search_queries` — the queries Gemini actually issued to Google Search.
*   `retrieval_queries` — the queries issued against your File Search store.

Together these tell you both *what* the model used and *how it looked for it*.

In [15]:
import json


def describe_grounding(response):
    """Dumps the interesting parts of grounding_metadata."""
    if response is None:
        print("No response returned.")
        return

    candidates = response.candidates or []
    if not candidates or candidates[0].grounding_metadata is None:
        print("No grounding metadata on this response.")
        return

    metadata = candidates[0].grounding_metadata

    print(f"web_search_queries: {metadata.web_search_queries}")
    print(f"retrieval_queries:  {metadata.retrieval_queries}")
    print(f"grounding_chunks:   {len(metadata.grounding_chunks or [])}")
    print(f"grounding_supports: {len(metadata.grounding_supports or [])}")

    print("\nFirst chunk, as JSON:")
    if metadata.grounding_chunks:
        print(json.dumps(
            metadata.grounding_chunks[0].model_dump(exclude_none=True),
            indent=2,
            default=str,
        )[:800])

    print("\nText spans and the chunks that support them:")
    for support in (metadata.grounding_supports or [])[:5]:
        segment_text = (support.segment.text or "")[:80]
        print(f"  {support.grounding_chunk_indices} <- \"{segment_text}\"")


if blended_response is not None:
    describe_grounding(blended_response)

## Clean up

File Search stores persist and count towards storage, so delete the store once you are done. The
local sample file goes away too.

In [16]:
import os

try:
    client.file_search_stores.delete(
        name=FILE_SEARCH_STORE_NAME,
        config=types.DeleteFileSearchStoreConfig(force=True),
    )
    print(f"Deleted store: {FILE_SEARCH_STORE_NAME}")
except Exception as e:
    print(f"Could not delete store: {type(e).__name__}: {e}")

if os.path.exists("sample_handbook.txt"):
    os.remove("sample_handbook.txt")
    print("Deleted local file: sample_handbook.txt")

print("Cleanup complete.")

Deleted store: fileSearchStores/zephyr-robotics-handbook-st-n0fqy4htnldi
Deleted local file: sample_handbook.txt
Cleanup complete.


## What's next

### Adapt this to your own documents

Swap the generated handbook for your real corpus and the rest of the notebook works unchanged:
upload each file with `upload_to_file_search_store`, and attach
[custom metadata](https://ai.google.dev/gemini-api/docs/file-search) such as team or document type
so you can narrow retrieval with a `metadata_filter`. If your fallback fires too eagerly, tighten
`is_sufficient` — requiring a minimum chunk count, or checking that the retrieved snippets share
keywords with the question, are both reasonable next steps before reaching for a scoring model.

### Learn more

*   The [File Search guide](https://ai.google.dev/gemini-api/docs/file-search) and its
    [pricing details](https://ai.google.dev/gemini-api/docs/file-search#pricing)
*   The [Grounding with Google Search guide](https://ai.google.dev/gemini-api/docs/google-search),
    including [inline citation attribution](https://ai.google.dev/gemini-api/docs/google-search#attributing_sources_with_inline_citations)
*   The [File Search quickstart](../quickstarts/File_Search.ipynb) for metadata filtering and
    multimodal stores
*   The [Grounding quickstart](../quickstarts/Grounding.ipynb) for Maps, YouTube, and URL context
*   The [Search grounding example](../examples/Search_grounding_for_research_report.ipynb) for a
    longer research-report workflow
*   More ideas in the [Gemini API Cookbook](https://github.com/google-gemini/cookbook)